In [0]:
sourceFolder = "/Volumes/main/default/fabmigration1_bronze/adventureworks2/"
targetEnv = "main.silver"
entityName = dbutils.widgets.get("EntityName")
tableName = entityName.replace(".", "_").lower()
fileExtension = "parquet"

In [0]:
%run /Workspace/Shared/Retail/00_Functions

In [0]:
# Set file path to load to DF
# Bronze
sourcePath = f"/Volumes/main/default/fabmigration1_bronze/adventureworks2/{entityName}.{fileExtension}"


In [0]:
# Read parquet file to get data\n"
dfdata = spark.read.load(sourcePath, format='parquet')

# Clean\n
dfdata = clean_df(dfdata)

# Set the schema for entity. Also if needed the first row.
entitySchema = GetSaleTableSchema(entityName)
firstRow = GetSaleTableFirstRow(entityName, dfdata)

# Create empty DF with schema and first row (if needed)
df = spark.createDataFrame(firstRow, entitySchema)

# Update column names from parquet file if needed
dfdata = UpdateDFColNames(entityName, dfdata)

# Join empty Df with data from parquet
df = df.unionByName(dfdata)

#display(df.limit(10))


In [0]:
# Set name for silver
entityNameFix = entityName.replace(".", "_").lower()

# Save DF into stage after clean process
SaveDF(df, targetEnv, entityNameFix)